In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DB = Path("../data/processed/oral_oncology.db")
conn = sqlite3.connect(DB)

patients   = pd.read_sql_query("SELECT * FROM patients", conn)
visits     = pd.read_sql_query("SELECT * FROM dental_visits", conn)
notes      = pd.read_csv("../data/processed/clinical_notes_enriched.csv")
referrals  = pd.read_sql_query("SELECT * FROM referrals", conn)
pathology  = pd.read_sql_query("SELECT * FROM pathology", conn)
appts      = pd.read_sql_query("SELECT * FROM appointments", conn)
outcomes   = pd.read_sql_query("SELECT * FROM outcomes", conn)
print("Loaded.")

Loaded.


In [2]:
visit_agg = visits.groupby("patient_id").agg(
    n_visits=("visit_id", "count"),
    n_lesion_visits=("lesion_present", "sum"),
    any_lesion=("lesion_present", "max"),
    last_visit_date=("visit_date", "max"),
    last_screening_date=("visit_date", lambda s: s[visits.loc[s.index, "screening_completed"] == 1].max() if (visits.loc[s.index, "screening_completed"] == 1).any() else None),
    screening_rate=("screening_completed", "mean"),
).reset_index()

ref_agg = referrals.groupby("patient_id").agg(
    n_referrals=("referral_id", "count"),
    n_referrals_completed=("referral_status", lambda s: (s == "Completed").sum()),
    n_referrals_pending=("referral_status", lambda s: (s == "Pending").sum()),
    n_referrals_noshow=("referral_status", lambda s: (s == "No-show").sum()),
    avg_days_to_specialist=("days_to_specialist_visit", "mean"),
).reset_index()

appt_agg = appts.groupby("patient_id").agg(
    n_appts=("appointment_id", "count"),
    no_show_rate=("no_show_flag", "mean"),
).reset_index()

nlp_agg = notes.groupby("patient_id").agg(
    max_nlp_urgency=("urgency_score", "max"),
    total_nlp_urgency=("urgency_score", "sum"),
    n_high_risk_notes=("note_risk_category", lambda s: (s == "High").sum()),
).reset_index()

path_agg = pathology.groupby("patient_id").agg(
    n_biopsies=("pathology_id", "count"),
    worst_dx=("diagnosis_result", "first"),
    worst_stage=("tumor_stage", "first"),
).reset_index()

# Master join
master = (patients
          .merge(visit_agg, on="patient_id", how="left")
          .merge(ref_agg,   on="patient_id", how="left")
          .merge(appt_agg,  on="patient_id", how="left")
          .merge(nlp_agg,   on="patient_id", how="left")
          .merge(path_agg,  on="patient_id", how="left")
          .merge(outcomes,  on="patient_id", how="left"))

# Fill NaN aggregates with sensible defaults
fill_zero = ["n_referrals", "n_referrals_completed", "n_referrals_pending",
             "n_referrals_noshow", "n_biopsies", "n_high_risk_notes",
             "max_nlp_urgency", "total_nlp_urgency"]
master[fill_zero] = master[fill_zero].fillna(0)

print(master.shape)
master.head()

(5000, 36)


,patient_id,age,sex,race_ethnicity,insurance_type,zip_code,smoking_status,alcohol_use,hpv_status,socioeconomic_risk_score,...,n_high_risk_notes,n_biopsies,worst_dx,worst_stage,diagnosis_status,diagnosis_stage,treatment_started,treatment_start_date,recurrence_status,survival_status
0,P00001,76,F,Hispanic,Medicaid,15236,Former,Light,Negative,0.348,...,0,0.0,NaN,NaN,Negative,NaN,0,NaN,0,Alive
1,P00002,66,F,White,Private,15238,Current,Heavy,Negative,0.168,...,0,0.0,NaN,NaN,Negative,NaN,0,NaN,0,Alive
2,P00003,61,F,White,Medicaid,15232,Never,No use,Negative,0.222,...,0,0.0,NaN,NaN,Negative,NaN,0,NaN,0,Alive
3,P00004,66,M,Hispanic,Medicare,15235,Never,No use,Negative,0.527,...,0,0.0,NaN,NaN,Negative,NaN,0,NaN,0,Alive
4,P00005,72,M,White,Medicaid,15235,Former,Light,Negative,0.484,...,0,0.0,NaN,NaN,Negative,NaN,0,NaN,0,Alive


In [3]:
from datetime import date

today = date(2026, 5, 1)
master["last_visit_date"] = pd.to_datetime(master["last_visit_date"])
master["last_screening_date"] = pd.to_datetime(master["last_screening_date"])

master["days_since_last_visit"] = (pd.Timestamp(today) - master["last_visit_date"]).dt.days
master["days_since_last_screening"] = (pd.Timestamp(today) - master["last_screening_date"]).dt.days

master["screening_overdue"] = ((master["days_since_last_screening"] > 365) |
                                master["last_screening_date"].isna()).astype(int)

master["age_group"] = pd.cut(master["age"],
                              bins=[0, 39, 49, 59, 69, 79, 120],
                              labels=["<40", "40-49", "50-59", "60-69", "70-79", "80+"])

master["risk_factors_count"] = (
    (master["smoking_status"].isin(["Current", "Former"])).astype(int) +
    (master["alcohol_use"].isin(["Moderate", "Heavy"])).astype(int) +
    (master["hpv_status"] == "Positive").astype(int) +
    (master["age"] >= 60).astype(int)
)

master["composite_risk_band"] = pd.cut(master["risk_factors_count"],
                                        bins=[-1, 0, 1, 2, 4],
                                        labels=["None", "Low", "Medium", "High"])

print(master[["age_group", "risk_factors_count", "composite_risk_band", "screening_overdue"]].head())

  age_group  risk_factors_count composite_risk_band  screening_overdue
0     70-79                   2              Medium                  0
1     60-69                   3                High                  0
2     60-69                   1                 Low                  0
3     60-69                   1                 Low                  0
4     70-79                   2              Medium                  1


In [4]:
out = Path("../data/processed/tableau_master.csv")
master.to_csv(out, index=False)
print(f"Saved {len(master)} rows to {out}")
print(f"File size: {out.stat().st_size / 1024:.1f} KB")

Saved 5000 rows to ../data/processed/tableau_master.csv
File size: 904.5 KB
